In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))
sys.path.insert(0, os.path.abspath("../sdlarch-rl"))


from sdlarch_rl.utils.utils import get_last_index, RealExcludeButtonsWrapper, GenericCNN, TimeLimit, FrameSkip
from sdlarch_rl import make
from stable_baselines3.common.atari_wrappers import WarpFrame
import time
import cv2
import numpy as np
import keyboard
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
from pathlib import Path
import pygame
from inputs import get_gamepad, devices
import threading
from imitation.data.types import Trajectory
import torch as th
import os
import re
from utils import get_last_index
import gymnasium as gym
from mario import make_env
from sbx import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecFrameStack

env = SubprocVecEnv(
    [make_env(11, human=False)]
)
env = VecFrameStack(env, 4, channels_order='last')

# --- CONFIGS ---
MAX_TRAJ = 10
SCALE = 1/4
SCREEN_WIDTH = 854 # int(640 * SCALE)
SCREEN_HEIGHT = 480 # int(480 * SCALE)
NUMBER_OF_ACTIONS = 6


# bc config
train_path = "./ppo/"
sub_train_path = train_path + "steps"
last_index_imitation = get_last_index(str(sub_train_path), "ppo_policy", "zip")
current_epoch = last_index_imitation


#TODO comment
# current_epoch = 48
latest_model_path = sub_train_path + f"/ppo_policy{current_epoch}.zip"
policy = PPO.load(latest_model_path, env=env)


class RendererThread(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True)
        self.img = None
        self.color = (0, 0, 255)
        self.count_record = 0
        self.running = True
        self.lock = threading.Lock()
        self.fps = 0

    def update_data(self, img, color, count, fps):
        with self.lock:
            self.img = img
            self.color = color
            self.count_record = count
            self.fps = fps

    def run(self):
        pygame.init()
        window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT), pygame.HWSURFACE | pygame.DOUBLEBUF)
        pygame.display.set_caption("New Super Mario Capture")
        font = pygame.font.SysFont("Arial", 24)
        
        while self.running:
            pygame.event.pump()
            
            img_to_render = None
            with self.lock:
                if self.img is not None:
                    img_to_render = self.img
                    current_color = self.color
                    current_count = self.count_record
                    current_fps = self.fps

            if img_to_render is not None:
                img_rgb = cv2.cvtColor(img_to_render, cv2.COLOR_BGR2RGB)
                surface = pygame.image.frombuffer(img_rgb.flatten(), (SCREEN_WIDTH, SCREEN_HEIGHT), 'RGB')
                
                window.blit(surface, (0, 0))

                
                fps_text = font.render(f"FPS: {int(current_fps)}", True, (255, 255, 255))
                count_text = font.render(f"Demos: {current_count}/{MAX_TRAJ}", True, (255, 255, 255))
                status_color = (0, 255, 0) if current_color == (0, 255, 0) else (255, 0, 0)
                rec_text = font.render("RECORDING" if current_color == (0, 255, 0) else "IDLE", True, status_color)
                
                window.blit(fps_text, (10, 10))
                window.blit(count_text, (10, 40))
                window.blit(rec_text, (10, 70))

                
                pygame.display.flip()
            
            time.sleep(1/30)


last_index = -1

gamepad = devices.gamepads[0]
STATE = {k: 0 for k in ["UP", "DOWN", "LEFT", "RIGHT", "A", "B", "X", "START", "CAM_X", "CAM_Y", "LT", "RT", "L3"]}
lock = threading.Lock()

DEADZONE = 10000
NORM = 32768


# Start render thread
renderer = RendererThread()
renderer.start()

is_running = False
is_recording = False
count_record = 0
trajectories = []
recorded_obs = []
recorded_actions = []

print("Done! Press 'K' to start/stop the record.")

fps_avg_frame_count = 0
fps_start_time = time.time()
actual_fps = 0

clock = pygame.time.Clock()

obs = env.reset()

total_rew = 0


while True:
    loop_start = time.time()
    
    pred_act, _ = policy.predict(obs, deterministic=True)
    # pred_act = [env.action_space.sample()]

    obs, rew, done, info = env.step(pred_act)


    total_rew += rew


    if done[0]:
        print(total_rew)
        env.reset()
        total_rew = 0

    # print(info)
    
    img = env.render()
    if obs is not None:
        # img = obs[0, :, :, -1]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized =  cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT), interpolation=cv2.INTER_NEAREST)
        color = (0, 255, 0) if is_running else (0, 0, 255)
        renderer.update_data(img_resized, color, count_record, actual_fps)

    
    fps_avg_frame_count += 1
    if time.time() - fps_start_time >= 1.0: # update fps every 1 second
        actual_fps = fps_avg_frame_count / (time.time() - fps_start_time)
        fps_avg_frame_count = 0
        fps_start_time = time.time()
        # print(f"FPS of capture: {actual_fps:.2f}")

    # try fix fps (25)
    clock.tick(60)

env.close()
renderer.running = False
pygame.quit()

D:\anaconda3\envs\env311\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Wrapping the env in a VecTransposeImage.
Done! Press 'K' to start/stop the record.
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
[-5.19452055]
